In [15]:
# Install Gymnasium with Box2D and Classic Control support
!pip install gymnasium[box2d,classic_control] moviepy

import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import random
from collections import deque, namedtuple
import matplotlib.pyplot as plt

# Check if GPU is available
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [16]:
class QNetwork(nn.Module):
    """Actor (Policy) Model."""
    def __init__(self, state_size, action_size, seed=42):
        super(QNetwork, self).__init__()
        self.seed = torch.manual_seed(seed)
        # Defining a standard architecture for LunarLander
        self.fc1 = nn.Linear(state_size, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, action_size)

    def forward(self, state):
        """Build a network that maps state -> action values."""
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

In [17]:
class ReplayBuffer:
    """Fixed-size buffer to store experience tuples."""
    def __init__(self, action_size, buffer_size, batch_size, seed):
        self.action_size = action_size
        self.memory = deque(maxlen=buffer_size)
        self.batch_size = batch_size
        self.experience = namedtuple("Experience", field_names=["state", "action", "reward", "next_state", "done"])
        self.seed = random.seed(seed)

    def add(self, state, action, reward, next_state, done):
        e = self.experience(state, action, reward, next_state, done)
        self.memory.append(e)

    def sample(self):
        experiences = random.sample(self.memory, k=self.batch_size)
        states = torch.from_numpy(np.vstack([e.state for e in experiences if e is not None])).float().to(device)
        actions = torch.from_numpy(np.vstack([e.action for e in experiences if e is not None])).long().to(device)
        rewards = torch.from_numpy(np.vstack([e.reward for e in experiences if e is not None])).float().to(device)
        next_states = torch.from_numpy(np.vstack([e.next_state for e in experiences if e is not None])).float().to(device)
        dones = torch.from_numpy(np.vstack([e.done for e in experiences if e is not None]).astype(np.uint8)).float().to(device)
        return (states, actions, rewards, next_states, dones)

    def __len__(self):
        return len(self.memory)

In [18]:
BUFFER_SIZE = int(1e5)  # Replay buffer size
BATCH_SIZE = 64         # Minibatch size
GAMMA = 0.99            # Discount factor
TAU = 1e-3              # For soft update of target parameters
LR = 5e-4               # Learning rate
UPDATE_EVERY = 4        # How often to update the network

class DQNAgent:
    def __init__(self, state_size, action_size, seed=42):
        self.state_size = state_size
        self.action_size = action_size
        self.seed = random.seed(seed)

        # Q-Network
        self.qnetwork_local = QNetwork(state_size, action_size, seed).to(device)
        self.qnetwork_target = QNetwork(state_size, action_size, seed).to(device)
        self.optimizer = optim.Adam(self.qnetwork_local.parameters(), lr=LR)

        # Replay memory
        self.memory = ReplayBuffer(action_size, BUFFER_SIZE, BATCH_SIZE, seed)
        self.t_step = 0

    def step(self, state, action, reward, next_state, done):
        # Save experience in replay memory
        self.memory.add(state, action, reward, next_state, done)

        # Learn every UPDATE_EVERY time steps
        self.t_step = (self.t_step + 1) % UPDATE_EVERY
        if self.t_step == 0:
            if len(self.memory) > BATCH_SIZE:
                return self.learn()
        return None

    def act(self, state, eps=0.):
        """Returns actions for given state as per current policy (Epsilon-greedy)."""
        state = torch.from_numpy(state).float().unsqueeze(0).to(device)
        self.qnetwork_local.eval()
        with torch.no_grad():
            action_values = self.qnetwork_local(state)
        self.qnetwork_local.train()

        # Exploration vs Exploitation [cite: 36]
        if random.random() > eps:
            return np.argmax(action_values.cpu().data.numpy())
        else:
            return random.choice(np.arange(self.action_size))

    def learn(self):
        # TODO 1 & 2: Sample batch and convert to tensors
        experiences = self.memory.sample()
        states, actions, rewards, next_states, dones = experiences

        # TODO 3: Get max predicted Q values from target model
        Q_targets_next = self.qnetwork_target(next_states).detach().max(1)[0].unsqueeze(1)

        # TODO 4: Compute Q targets for current states
        Q_targets = rewards + (GAMMA * Q_targets_next * (1 - dones))

        # TODO 5: Get expected Q values from local model
        Q_expected = self.qnetwork_local(states).gather(1, actions)

        # TODO 6: Compute loss, backpropagate, and update
        loss = F.mse_loss(Q_expected, Q_targets)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        # TODO 7: Soft Update the Target Network
        self.soft_update(self.qnetwork_local, self.qnetwork_target, TAU)

        return loss.item()

    def soft_update(self, local_model, target_model, tau):
        """Soft update model parameters: θ_target = τ*θ_local + (1 - τ)*θ_target"""
        for target_param, local_param in zip(target_model.parameters(), local_model.parameters()):
            target_param.data.copy_(tau*local_param.data + (1.0-tau)*target_param.data)

In [ ]:
def train(n_episodes=1000):
    env = gym.make("LunarLander-v3")
    agent = DQNAgent(state_size=8, action_size=4)

    scores = []
    losses = []
    eps = 1.0
    eps_end = 0.01
    eps_decay = 0.995

    for i_episode in range(1, n_episodes + 1):
        state, _ = env.reset()
        score = 0
        episode_losses = []

        for t in range(1000):
            action = agent.act(state, eps)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            loss = agent.step(state, action, reward, next_state, done)
            if loss is not None:
                episode_losses.append(loss)

            state = next_state
            score += reward
            if done:
                break

        scores.append(score)
        avg_loss = np.mean(episode_losses) if episode_losses else 0.0
        losses.append(avg_loss)
        eps = max(eps_end, eps_decay * eps) # Decay epsilon

        if i_episode % 100 == 0:
            print(f'Episode {i_episode}\tAverage Score: {np.mean(scores[-100:]):.2f}')

    return scores, losses, agent

# Execute Training
scores, losses, agent = train(n_episodes=1000)

# Figure 1: Rewards
plt.figure(figsize=(10,5))
plt.plot(scores)
plt.title("Learning Progress: Rewards")
plt.xlabel("Episode")
plt.ylabel("Total Reward")
plt.show()

# Figure 2: Loss
plt.figure(figsize=(10,5))
plt.plot(losses, color='orange')
plt.title("Learning Progress: Training Loss")
plt.xlabel("Episode")
plt.ylabel("Average Loss")
plt.show()

Episode 100	Average Score: -147.38
Episode 200	Average Score: -110.69
Episode 300	Average Score: -44.36


In [20]:
import gymnasium as gym
from gymnasium.wrappers import RecordVideo

def record_video(agent, video_folder='videos', name_prefix='lunar_lander'):
    """Records a video of the trained agent performing in the environment."""
    # 1. Initialize the environment with rgb_array render mode
    env = gym.make("LunarLander-v2", render_mode="rgb_array")

    # 2. Wrap the environment with RecordVideo
    # We use episode_trigger=lambda x: True to ensure it records the single episode we run
    env = RecordVideo(env, video_folder=video_folder, name_prefix=name_prefix, episode_trigger=lambda x: True)

    state, _ = env.reset()
    done = False

    while not done:
        # Use the trained agent to pick the best action (exploitation: eps = 0)
        action = agent.act(state, eps=0.0)
        state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

    # Close the environment to properly save the mp4 file
    env.close()
    print(f"Video saved to the '{video_folder}' directory.")

# Run this after training is complete
# record_video(agent)

In [21]:
# Save the weights of the local Q-Network
torch.save(agent.qnetwork_local.state_dict(), 'lunar_lander_model.pth')

NameError: name 'agent' is not defined